In [1]:
import os
import math
import random
from collections import deque

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
os.environ["PYTHONHASHSEED"]="42"

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use("dark_background")

MODEL_PATH="saved_models"
FIGURE_PATH="figures"
RESULT_PATH="results"

os.makedirs(MODEL_PATH,exist_ok=True)
os.makedirs(FIGURE_PATH,exist_ok=True)
os.makedirs(RESULT_PATH,exist_ok=True)

In [3]:
S0=100.0
K=100.0
T=1.0
r=0.05
sigma=0.25
steps=50

gamma=0.99
learning_rate=0.001
batch_size=64
buffer_size=50000

episodes=40000
evaluation_episodes=10000

epsilon=1.0
epsilon_min=0.01
epsilon_decay=0.9995

target_update=200

In [4]:
print("="*60)
print("American Put Option Pricing using Deep Q Network")
print("="*60)

print(f"TensorFlow Version       : {tf.__version__}")
print(f"Random Seed             : 42")
print(f"Initial Stock Price     : {S0}")
print(f"Strike Price            : {K}")
print(f"Time to Maturity        : {T}")
print(f"Risk-Free Rate          : {r}")
print(f"Volatility              : {sigma}")
print(f"Binomial Steps          : {steps}")
print(f"Training Episodes       : {episodes}")
print(f"Evaluation Episodes     : {evaluation_episodes}")
print(f"Learning Rate           : {learning_rate}")
print(f"Discount Factor         : {gamma}")
print(f"Batch Size              : {batch_size}")
print(f"Replay Buffer Size      : {buffer_size}")
print(f"Target Update Frequency : {target_update}")
print(f"Epsilon Decay           : {epsilon_decay}")
print(f"GPU Available           : {len(tf.config.list_physical_devices('GPU'))>0}")

print("="*60)

American Put Option Pricing using Deep Q Network
TensorFlow Version       : 2.21.0
Random Seed             : 42
Initial Stock Price     : 100.0
Strike Price            : 100.0
Time to Maturity        : 1.0
Risk-Free Rate          : 0.05
Volatility              : 0.25
Binomial Steps          : 50
Training Episodes       : 40000
Evaluation Episodes     : 10000
Learning Rate           : 0.001
Discount Factor         : 0.99
Batch Size              : 64
Replay Buffer Size      : 50000
Target Update Frequency : 200
Epsilon Decay           : 0.9995
GPU Available           : False


In [5]:
def crr_put_price(
        S0: float,
        K: float,
        T: float,
        r: float,
        sigma: float,
        steps: int,
        american: bool
        ):

    if S0<=0 or K<=0:
        raise ValueError("S0 and K must be positive")

    if T<=0:
        return max(K-S0,0.0)

    if sigma<=0:
        raise ValueError("sigma must be positive")

    if int(steps)!=steps or steps<1:
        raise ValueError("steps must be a positive integer")

    steps=int(steps)

    dt=T/steps
    u=math.exp(sigma*math.sqrt(dt))
    d=1.0/u

    growth=math.exp(r*dt)
    p=(growth-d)/(u-d)
    disc=math.exp(-r*dt)

    if not (0.0<p<1.0):
        raise ValueError("invalid risk neutral probability")

    j=np.arange(steps+1)
    stock=S0*(u**j)*(d**(steps-j))
    value=np.maximum(K-stock,0.0)

    for i in range(steps-1,-1,-1):

        value=disc*(p*value[1:i+2]+(1.0-p)*value[0:i+1])

        if american:
            j=np.arange(i+1)
            stock=S0*(u**j)*(d**(i-j))
            exercise=np.maximum(K-stock,0.0)
            value=np.maximum(value,exercise)

    return float(value[0])

In [6]:
class AmericanPutEnv:

    HOLD=0
    EXERCISE=1

    def __init__(self,S0,K,T,r,sigma,steps):

        self.S0=S0
        self.K=K
        self.T=T
        self.r=r
        self.sigma=sigma
        self.steps=steps

        self.dt=T/steps
        self.u=math.exp(sigma*math.sqrt(self.dt))
        self.d=1.0/self.u
        self.p=(math.exp(r*self.dt)-self.d)/(self.u-self.d)
        self.discount=math.exp(-r*self.dt)

        self.reset()

    def get_state(self):

        return np.array([
            self.current_step/self.steps,
            self.stock_price/self.K
        ],dtype=np.float32)

    def reset(self):

        self.current_step=0
        self.stock_price=self.S0
        self.done=False

        return self.get_state()

    def step(self,action):

        if self.done:
            return self.get_state(),0.0,True

        if action==self.EXERCISE:

            reward=max(self.K-self.stock_price,0.0)
            self.done=True

            return self.get_state(),reward,True

        self.current_step+=1

        if np.random.rand()<self.p:
            self.stock_price*=self.u
        else:
            self.stock_price*=self.d

        if self.current_step>=self.steps:

            reward=max(self.K-self.stock_price,0.0)
            self.done=True

            return self.get_state(),reward,True

        return self.get_state(),0.0,False

In [7]:
replay_buffer=deque(maxlen=buffer_size)

In [8]:
state_size=2
action_size=2

model=tf.keras.Sequential()

model.add(tf.keras.layers.Input(shape=(state_size,)))
model.add(tf.keras.layers.Dense(128,activation="relu"))
model.add(tf.keras.layers.Dense(128,activation="relu"))
model.add(tf.keras.layers.Dense(64,activation="relu"))
model.add(tf.keras.layers.Dense(action_size,activation="linear"))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss="mse"
)

target_model=tf.keras.models.clone_model(model)
target_model.set_weights(model.get_weights())

In [9]:
CHECKPOINT_PATH=os.path.join(MODEL_PATH,"checkpoint.weights.h5")
STATE_PATH=os.path.join(RESULT_PATH,"training_state.npz")

Checkpoint Saved : Episode 500
Checkpoint Saved : Episode 1000
Episode 1000/40000 | Epsilon 0.2833 | Average Reward 2.5669 | Average Loss 0.241471
